# 20 - Offline Reinforcement Learning

## Learning Objectives
1. Collect an offline dataset with a behavior policy and identify coverage gaps
2. Implement CQL vs standard Q-learning and show Q-value divergence vs stability
3. Demonstrate behavioral cloning baseline and dataset coverage analysis
4. Show online fine-tuning from CQL-pretrained policy vs from-scratch training


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Packages loaded: numpy', np.__version__)
print('Offline RL: GridWorld dataset collection, CQL, BC, online fine-tuning')


## Level 1: Offline Dataset Collection and Coverage Analysis

Collect a 5x5 GridWorld dataset using a behavior policy with epsilon=0.5.
Visualize state visitation frequency: sparse coverage creates extrapolation risk.
Coverage gaps are why offline RL is harder than online RL.


In [ ]:
class GridWorld:
    """5x5 GridWorld: start=(0,0), goal=(4,4). 4 actions: up/down/left/right."""

    def __init__(self, rows: int = 5, cols: int = 5):
        self.rows = rows
        self.cols = cols
        self.goal = (rows - 1, cols - 1)
        self.n_states = rows * cols
        self.n_actions = 4  # 0=up, 1=down, 2=left, 3=right
        self.deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def reset(self):
        self.pos = (0, 0)
        return self._state_idx()

    def _state_idx(self) -> int:
        return self.pos[0] * self.cols + self.pos[1]

    def step(self, action: int) -> tuple:
        r, c = self.pos
        dr, dc = self.deltas[action]
        nr = max(0, min(self.rows - 1, r + dr))
        nc = max(0, min(self.cols - 1, c + dc))
        self.pos = (nr, nc)
        done = self.pos == self.goal
        reward = 1.0 if done else -0.01
        return self._state_idx(), reward, done

    def optimal_action(self, s: int) -> int:
        """Greedy: minimize Manhattan distance to goal."""
        r, c = s // self.cols, s % self.cols
        gr, gc = self.goal
        best_a, best_dist = 0, 9999
        for a, (dr, dc) in enumerate(self.deltas):
            nr = max(0, min(self.rows - 1, r + dr))
            nc = max(0, min(self.cols - 1, c + dc))
            d = abs(nr - gr) + abs(nc - gc)
            if d < best_dist:
                best_dist = d
                best_a = a
        return best_a


def collect_offline_dataset(
    env: GridWorld, n_episodes: int = 300, epsilon: float = 0.5
) -> tuple:
    """Collect (s, a, r, s', done) tuples using epsilon-greedy behavior policy.
    Higher epsilon = more random actions = more coverage but lower quality data.
    """
    dataset = []
    state_visits = np.zeros(env.n_states, dtype=int)
    for ep in range(n_episodes):
        s = env.reset()
        done = False
        steps = 0
        while not done and steps < 50:
            a = (np.random.randint(env.n_actions) if np.random.random() < epsilon
                  else env.optimal_action(s))
            s_next, r, done = env.step(a)
            dataset.append((s, a, r, s_next, done))
            state_visits[s] += 1
            s = s_next
            steps += 1
    return dataset, state_visits


# Collect offline dataset
np.random.seed(42)
env = GridWorld(5, 5)
offline_dataset, state_visits = collect_offline_dataset(
    env, n_episodes=300, epsilon=0.5
)

print(f'Dataset size: {len(offline_dataset)} transitions')
print(f'States visited: {np.sum(state_visits > 0)} / {env.n_states}')
print(f'Unvisited states: {np.sum(state_visits == 0)}')

# Count (s, a) pairs
sa_counts = np.zeros((env.n_states, env.n_actions), dtype=int)
for s, a, r, sn, d in offline_dataset:
    sa_counts[s, a] += 1
print(f'State-action pairs missing from data: '
      f'{np.sum(sa_counts == 0)} / {env.n_states * env.n_actions}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

visit_map = state_visits.reshape(env.rows, env.cols)
im0 = axes[0].imshow(visit_map, cmap='Blues', interpolation='nearest')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('State Visitation Frequency\n(behavior eps=0.5)', fontsize=11)
for i in range(env.rows):
    for j in range(env.cols):
        v = visit_map[i, j]
        axes[0].text(j, i, str(v), ha='center', va='center', fontsize=9,
                     color='white' if v > visit_map.max() / 2 else 'black')

# Coverage ratio vs epsilon
epsilons_test = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
coverage_ratios = []
for eps_t in epsilons_test:
    np.random.seed(0)
    _, sv = collect_offline_dataset(env, n_episodes=200, epsilon=eps_t)
    coverage_ratios.append(np.sum(sv > 0) / env.n_states)

axes[1].plot(epsilons_test, coverage_ratios, 'o-', color='#d7191c',
             linewidth=2, markersize=8)
axes[1].axhline(y=1.0, color='#1a9641', linestyle='--', label='Full coverage')
axes[1].set_xlabel('Behavior Policy Epsilon', fontsize=12)
axes[1].set_ylabel('State Coverage Ratio', fontsize=12)
axes[1].set_title('Coverage vs Behavior Policy Randomness', fontsize=12)
axes[1].legend(); axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('/tmp/offline_rl_coverage.png', dpi=100, bbox_inches='tight')
plt.show()


## Level 2: CQL vs Standard Q-Learning on Offline Dataset

Standard offline Q-learning overestimates Q for unseen (s,a) pairs
because the max-Q Bellman backup bootstraps from OOD Q-values.
CQL adds: alpha * (E_{s,a~pi}[Q] - E_{s,a~D}[Q])
This actively pushes OOD Q-values down toward zero, preventing divergence.


In [ ]:
def get_batch(dataset: list, batch_size: int = 64) -> tuple:
    """Sample random mini-batch from offline dataset."""
    idx = np.random.choice(len(dataset), batch_size, replace=False)
    batch = [dataset[i] for i in idx]
    s = np.array([t[0] for t in batch])
    a = np.array([t[1] for t in batch])
    r = np.array([t[2] for t in batch])
    s_next = np.array([t[3] for t in batch])
    done = np.array([t[4] for t in batch], dtype=float)
    return s, a, r, s_next, done


def train_standard_q(
    dataset: list, n_states: int, n_actions: int,
    n_updates: int = 800, alpha: float = 0.2,
    gamma: float = 0.9, batch_size: int = 32,
) -> tuple:
    """Standard offline Q-learning (no conservatism).
    Overestimates Q(s, a) for state-action pairs not seen in data.
    """
    Q = np.zeros((n_states, n_actions))
    q_max_history = []
    for _ in range(n_updates):
        s, a, r, s_next, done = get_batch(dataset, batch_size)
        # Bellman update: max over ALL actions at s_next (includes OOD)
        td_target = r + gamma * (1 - done) * Q[s_next].max(axis=1)
        for i in range(len(s)):
            Q[s[i], a[i]] += alpha * (td_target[i] - Q[s[i], a[i]])
        q_max_history.append(Q.max())
    return Q, q_max_history


def train_cql(
    dataset: list, n_states: int, n_actions: int,
    n_updates: int = 800, alpha: float = 0.2,
    gamma: float = 0.9, cql_alpha: float = 0.3,
    batch_size: int = 32,
) -> tuple:
    """Conservative Q-Learning (CQL) for offline RL.
    For each update:
      1. Standard Bellman update for data (s, a) pair.
      2. Conservative penalty: pull OOD Q-values (never seen in data) toward zero.
    cql_alpha controls how aggressively OOD actions are penalized.
    """
    Q = np.zeros((n_states, n_actions))
    q_max_history = []
    ood_q_history = []  # track mean OOD Q value (should stay near 0)

    # Precompute which (s, a) pairs are in dataset
    in_dataset = np.zeros((n_states, n_actions), dtype=bool)
    for s_d, a_d, _, _, _ in dataset:
        in_dataset[s_d, a_d] = True

    for _ in range(n_updates):
        s, a, r, s_next, done = get_batch(dataset, batch_size)

        # Bellman update for data actions
        td_target = r + gamma * (1 - done) * Q[s_next].max(axis=1)
        for i in range(len(s)):
            Q[s[i], a[i]] += alpha * (td_target[i] - Q[s[i], a[i]])

        # CQL: penalize OOD Q values at visited states
        # Pull unseen (s, a) Q-values toward zero
        unique_states = np.unique(s)
        for si in unique_states:
            for ai in range(n_actions):
                if not in_dataset[si, ai]:
                    # Decay OOD Q toward zero
                    Q[si, ai] *= (1.0 - cql_alpha)

        q_max_history.append(Q.max())
        ood_mask = ~in_dataset
        ood_q_history.append(float(Q[ood_mask].mean()))

    return Q, q_max_history, ood_q_history


# To show divergence clearly, use a SMALL sparse dataset
np.random.seed(42)
sparse_dataset, _ = collect_offline_dataset(env, n_episodes=20, epsilon=0.3)
print(f'Sparse dataset: {len(sparse_dataset)} transitions')

print('Training standard Q-learning on sparse dataset...')
np.random.seed(42)
Q_std_sparse, q_max_std = train_standard_q(
    sparse_dataset, env.n_states, env.n_actions, n_updates=800, alpha=0.2
)

print('Training CQL on sparse dataset...')
np.random.seed(42)
Q_cql_sparse, q_max_cql, ood_q_cql = train_cql(
    sparse_dataset, env.n_states, env.n_actions,
    n_updates=800, alpha=0.2, cql_alpha=0.3
)

# Also train both on full offline_dataset for policy comparison
np.random.seed(42)
Q_std, _ = train_standard_q(offline_dataset, env.n_states, env.n_actions,
                              n_updates=500, alpha=0.1)
np.random.seed(42)
Q_cql, _, _ = train_cql(offline_dataset, env.n_states, env.n_actions,
                          n_updates=500, alpha=0.1, cql_alpha=0.1)

print(f'Sparse - Standard Q max: {max(q_max_std):.4f}')
print(f'Sparse - CQL Q max:      {max(q_max_cql):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(q_max_std, color='#d7191c', linewidth=2, label='Standard Q-learning')
axes[0].plot(q_max_cql, color='#2c7bb6', linewidth=2, label='CQL')
axes[0].axhline(y=1.0, color='#1a9641', linestyle='--', alpha=0.7,
                label='True Q* max (approx 1.0)')
axes[0].set_xlabel('Training Update', fontsize=12)
axes[0].set_ylabel('Max Q Value', fontsize=12)
axes[0].set_title('Q-value Stability: Standard QL vs CQL\n(sparse offline data)', fontsize=11)
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

axes[1].plot(ood_q_cql, color='#1a9641', linewidth=2)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5,
                label='Zero (ideal OOD Q)')
axes[1].set_xlabel('Training Update', fontsize=12)
axes[1].set_ylabel('Mean Q(s, OOD action)', fontsize=12)
axes[1].set_title('CQL OOD Q Suppression\n(unseen actions stay near zero)', fontsize=11)
axes[1].legend(); axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('/tmp/offline_rl_cql_vs_std.png', dpi=100, bbox_inches='tight')
plt.show()


## Real-World Example 1: Behavioral Cloning Baseline

BC clones the behavior policy: majority-vote action per state from offline data.
BC = supervised learning on (s, a) pairs: no reward signal, no Bellman backup.
Compare: BC vs CQL vs standard Q vs optimal on success rate and mean return.


In [ ]:
def train_bc(dataset: list, n_states: int, n_actions: int) -> np.ndarray:
    """Behavioral Cloning: majority-vote action per visited state.
    States not in dataset get default action 0 (up).
    """
    action_counts = np.zeros((n_states, n_actions), dtype=float)
    for s, a, r, sn, d in dataset:
        action_counts[s, a] += 1
    # Majority vote: argmax over observed actions
    bc_policy = np.argmax(action_counts, axis=1)
    return bc_policy


def evaluate_policy(
    Q: np.ndarray, env: GridWorld,
    n_episodes: int = 200, max_steps: int = 50,
    policy_array: np.ndarray = None,
) -> tuple:
    """Evaluate a Q-table (argmax) or BC array policy.
    Returns (success_rate, mean_return).
    """
    success_count = 0
    total_returns = []
    for _ in range(n_episodes):
        s = env.reset()
        done = False
        total_r = 0.0
        for _ in range(max_steps):
            a = int(policy_array[s]) if policy_array is not None else int(np.argmax(Q[s]))
            s, r, done = env.step(a)
            total_r += r
            if done:
                break
        success_count += int(done)
        total_returns.append(total_r)
    return success_count / n_episodes, float(np.mean(total_returns))


def evaluate_optimal(env: GridWorld, n_episodes: int = 200,
                      max_steps: int = 50) -> tuple:
    """Oracle optimal policy performance."""
    success, returns = 0, []
    for _ in range(n_episodes):
        s = env.reset()
        done = False
        total_r = 0.0
        for _ in range(max_steps):
            a = env.optimal_action(s)
            s, r, done = env.step(a)
            total_r += r
            if done:
                break
        success += int(done)
        returns.append(total_r)
    return success / n_episodes, float(np.mean(returns))


# Train BC
bc_policy = train_bc(offline_dataset, env.n_states, env.n_actions)

# Evaluate all methods
opt_succ, opt_ret = evaluate_optimal(env)
cql_succ, cql_ret = evaluate_policy(Q_cql, env)
std_succ, std_ret = evaluate_policy(Q_std, env)
bc_succ, bc_ret = evaluate_policy(None, env, policy_array=bc_policy)

print('Policy Evaluation Results:')
print(f'{"Method":<20} {"Success Rate":<15} {"Mean Return":<12}')
print('-' * 47)
for name, succ, ret in [
    ('Optimal', opt_succ, opt_ret),
    ('CQL (full data)', cql_succ, cql_ret),
    ('Standard QL', std_succ, std_ret),
    ('BC', bc_succ, bc_ret),
]:
    print(f'{name:<20} {succ:<15.3f} {ret:<12.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
methods_eval = ['Optimal', 'CQL', 'Std QL', 'BC']
success_rates = [opt_succ, cql_succ, std_succ, bc_succ]
mean_returns = [opt_ret, cql_ret, std_ret, bc_ret]
colors_eval = ['#1a9641', '#2c7bb6', '#d7191c', '#fd8d3c']

bars0 = axes[0].bar(methods_eval, success_rates, color=colors_eval,
                     alpha=0.85, edgecolor='black')
axes[0].set_ylabel('Success Rate', fontsize=12)
axes[0].set_title('Policy Comparison: Success Rate', fontsize=12)
axes[0].set_ylim(0, 1.15)
axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4, axis='y')
for bar, val in zip(bars0, success_rates):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.02, f'{val:.2f}',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

bars1 = axes[1].bar(methods_eval, mean_returns, color=colors_eval,
                     alpha=0.85, edgecolor='black')
axes[1].set_ylabel('Mean Episode Return', fontsize=12)
axes[1].set_title('Policy Comparison: Mean Return', fontsize=12)
axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4, axis='y')
for bar, val in zip(bars1, mean_returns):
    y_off = max(0, bar.get_height()) + 0.005
    axes[1].text(bar.get_x() + bar.get_width() / 2, y_off,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/offline_rl_bc_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## Real-World Example 2: Coverage Analysis and Q-value Extrapolation Error

Compute oracle Q* via full value iteration on the known GridWorld.
Measure: |Q_learned(s,a) - Q*(s,a)| for visited vs unvisited state-action pairs.
Standard Q overestimates OOD Q values; CQL suppresses them toward zero.


In [ ]:
def compute_true_q(env: GridWorld, gamma: float = 0.9, n_iter: int = 100) -> np.ndarray:
    """Oracle Q* via value iteration on known GridWorld dynamics."""
    Q_true = np.zeros((env.n_states, env.n_actions))

    def next_state(s, a):
        r, c = s // env.cols, s % env.cols
        dr, dc = env.deltas[a]
        nr = max(0, min(env.rows - 1, r + dr))
        nc = max(0, min(env.cols - 1, c + dc))
        return nr * env.cols + nc

    for _ in range(n_iter):
        Q_new = np.zeros_like(Q_true)
        for s in range(env.n_states):
            for a in range(env.n_actions):
                ns = next_state(s, a)
                done = (ns // env.cols, ns % env.cols) == env.goal
                rew = 1.0 if done else -0.01
                Q_new[s, a] = rew + gamma * (0.0 if done else Q_true[ns].max())
        Q_true = Q_new
    return Q_true


Q_true = compute_true_q(env)
print(f'True Q* max: {Q_true.max():.4f}')

# Identify visited vs unvisited (s, a) pairs
in_dataset = np.zeros((env.n_states, env.n_actions), dtype=bool)
for s, a, r, sn, d in offline_dataset:
    in_dataset[s, a] = True

visited_pairs = in_dataset.flatten()
ood_pairs = ~visited_pairs

# Extrapolation error: |Q_learned - Q*| for in-distribution vs OOD
def extrapolation_stats(Q, label):
    q_flat = Q.flatten()
    qt_flat = Q_true.flatten()
    err_vis = np.mean(np.abs(q_flat[visited_pairs] - qt_flat[visited_pairs]))
    err_ood = np.mean(np.abs(q_flat[ood_pairs] - qt_flat[ood_pairs]))
    overest_vis = np.mean(q_flat[visited_pairs] > qt_flat[visited_pairs])
    overest_ood = np.mean(q_flat[ood_pairs] > qt_flat[ood_pairs])
    print(f'{label}:')
    print(f'  In-dist MAE: {err_vis:.4f} | OOD MAE: {err_ood:.4f}')
    print(f'  In-dist overestimation: {overest_vis:.2f} | OOD overestimation: {overest_ood:.2f}')
    return err_vis, err_ood, overest_vis, overest_ood


ev_std, eo_std, ov_std, oo_std = extrapolation_stats(Q_std, 'Standard Q')
ev_cql, eo_cql, ov_cql, oo_cql = extrapolation_stats(Q_cql, 'CQL')

# Visualize: scatter Q_learned vs Q_true for visited and OOD pairs
q_true_flat = Q_true.flatten()
q_std_flat = Q_std.flatten()
q_cql_flat = Q_cql.flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, q_flat, title in [
    (axes[0], q_std_flat, 'Standard Q: Q_learned vs Q*'),
    (axes[1], q_cql_flat, 'CQL: Q_learned vs Q*'),
]:
    ax.scatter(q_true_flat[visited_pairs], q_flat[visited_pairs],
               color='#2c7bb6', alpha=0.6, s=40, label='In-dataset', zorder=5)
    if ood_pairs.any():
        ax.scatter(q_true_flat[ood_pairs], q_flat[ood_pairs],
                   color='#d7191c', alpha=0.6, s=40, marker='x',
                   linewidths=1.5, label='OOD (unseen)', zorder=6)
    min_v = min(q_true_flat.min(), q_flat.min())
    max_v = max(q_true_flat.max(), q_flat.max())
    ax.plot([min_v, max_v], [min_v, max_v], 'k--', alpha=0.4,
            label='Perfect estimate')
    ax.set_xlabel('True Q*(s,a)', fontsize=12)
    ax.set_ylabel('Learned Q(s,a)', fontsize=12)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9); ax.set_facecolor('#f8f8f8'); ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('/tmp/offline_rl_extrapolation.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'OOD overestimation rate - Std: {oo_std:.2f}, CQL: {oo_cql:.2f}')


## Real-World Example 3: Online Fine-Tuning from CQL Pretrained Policy

Start from CQL-pretrained Q-table, add 500 online steps of interaction.
Compare: (1) CQL pretrained + fine-tune vs (2) online from-scratch Q-learning.
CQL pretraining provides a warm start, requiring fewer samples to reach optimal.


In [ ]:
def online_q_learning(
    env: GridWorld, Q_init: np.ndarray,
    n_steps: int = 500, alpha: float = 0.1,
    gamma: float = 0.9, eps: float = 0.2,
    max_ep_steps: int = 50,
) -> tuple:
    """Online Q-learning from Q_init. Returns (Q_final, episode_success_list)."""
    Q = Q_init.copy()
    success_history = []
    total_steps = 0

    while total_steps < n_steps:
        s = env.reset()
        done = False
        t = 0
        while not done and t < max_ep_steps and total_steps < n_steps:
            a = (np.random.randint(env.n_actions) if np.random.random() < eps
                  else int(np.argmax(Q[s])))
            s_next, r, done = env.step(a)
            td_target = r + gamma * (0.0 if done else Q[s_next].max())
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s_next
            t += 1
            total_steps += 1
        success_history.append(1.0 if done else 0.0)

    return Q, success_history


N_ONLINE_STEPS = 500

# Fine-tune from CQL pretrained
np.random.seed(42)
Q_ft, ft_success = online_q_learning(
    env, Q_cql.copy(), n_steps=N_ONLINE_STEPS, eps=0.1
)

# Train from scratch
np.random.seed(42)
Q_scratch, scratch_success = online_q_learning(
    env, np.zeros((env.n_states, env.n_actions)),
    n_steps=N_ONLINE_STEPS, eps=0.2
)

ft_succ_final, ft_ret_final = evaluate_policy(Q_ft, env)
scratch_succ_final, scratch_ret_final = evaluate_policy(Q_scratch, env)

print(f'Online fine-tuning from CQL ({N_ONLINE_STEPS} steps):')
print(f'  Success rate: {ft_succ_final:.3f}, Mean return: {ft_ret_final:.4f}')
print(f'From-scratch online ({N_ONLINE_STEPS} steps):')
print(f'  Success rate: {scratch_succ_final:.3f}, Mean return: {scratch_ret_final:.4f}')

def rolling_mean(arr, w):
    return np.convolve(arr, np.ones(w) / w, mode='valid')

window = 10
ft_ma = rolling_mean(ft_success, window)
scratch_ma = rolling_mean(scratch_success, window)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(ft_ma, color='#2c7bb6', linewidth=2, label='CQL pretrained + online FT')
axes[0].plot(scratch_ma, color='#d7191c', linewidth=2, linestyle='--',
             label='From-scratch online')
axes[0].axhline(y=opt_succ, color='#1a9641', linestyle=':',
                label=f'Optimal ({opt_succ:.2f})')
axes[0].set_xlabel('Online Episode', fontsize=12)
axes[0].set_ylabel('Success Rate (rolling avg)', fontsize=12)
axes[0].set_title(f'Online Fine-Tuning ({N_ONLINE_STEPS} steps):\nCQL Init vs From Scratch', fontsize=11)
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

# Full summary bar chart
all_methods = ['Optimal', 'CQL+FT', 'CQL only', 'Std QL', 'BC', 'Scratch']
all_success = [opt_succ, ft_succ_final, cql_succ, std_succ, bc_succ, scratch_succ_final]
all_colors = ['#1a9641', '#2c7bb6', '#abd9e9', '#d7191c', '#fd8d3c', '#636363']
bars = axes[1].bar(all_methods, all_success, color=all_colors,
                    alpha=0.85, edgecolor='black')
axes[1].set_ylabel('Success Rate', fontsize=12)
axes[1].set_title('All Methods: Final Policy Quality', fontsize=12)
axes[1].set_ylim(0, 1.15)
axes[1].set_xticklabels(all_methods, rotation=15, ha='right', fontsize=9)
axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4, axis='y')
for bar, val in zip(bars, all_success):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/offline_rl_finetuning.png', dpi=100, bbox_inches='tight')
plt.show()


## Comparison: BC vs Standard Q vs CQL on Offline GridWorld

CQL alpha sweep: too low = no conservatism (like standard Q), too high = over-conservative (like BC).


In [ ]:
# CQL alpha sweep: show tradeoff between conservatism and performance
cql_alphas = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
cql_success_sweep = []
cql_q_max_sweep = []

for alpha_cql in cql_alphas:
    np.random.seed(42)
    if alpha_cql == 0.0:
        Q_tmp, q_h = train_standard_q(
            offline_dataset, env.n_states, env.n_actions, n_updates=500, alpha=0.1
        )
        q_max_final = float(max(q_h))
    else:
        Q_tmp, q_h, _ = train_cql(
            offline_dataset, env.n_states, env.n_actions,
            n_updates=500, alpha=0.1, cql_alpha=alpha_cql
        )
        q_max_final = float(max(q_h))
    succ, _ = evaluate_policy(Q_tmp, env)
    cql_success_sweep.append(float(succ))
    cql_q_max_sweep.append(q_max_final)

print('CQL Alpha Sweep:')
print(f'{"cql_alpha":<12} {"Success":<10} {"MaxQ":<10}')
print('-' * 32)
for a, s, q in zip(cql_alphas, cql_success_sweep, cql_q_max_sweep):
    print(f'{a:<12.2f} {s:<10.3f} {q:<10.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(cql_alphas, cql_q_max_sweep, 'o-', color='#d7191c',
             linewidth=2, markersize=8)
axes[0].axhline(y=Q_true.max(), color='#1a9641', linestyle='--',
                label=f'True Q* max ({Q_true.max():.2f})')
axes[0].set_xlabel('CQL Alpha (conservatism strength)', fontsize=12)
axes[0].set_ylabel('Max Q Value', fontsize=12)
axes[0].set_title('CQL: Max Q vs Alpha\n(0=no conservatism, high=over-conservative)', fontsize=11)
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

axes[1].plot(cql_alphas, cql_success_sweep, 's-', color='#2c7bb6',
             linewidth=2, markersize=8)
axes[1].axhline(y=bc_succ, color='#fd8d3c', linestyle='--',
                label=f'BC ({bc_succ:.2f})')
axes[1].axhline(y=opt_succ, color='#1a9641', linestyle=':',
                label=f'Optimal ({opt_succ:.2f})')
axes[1].set_xlabel('CQL Alpha', fontsize=12)
axes[1].set_ylabel('Success Rate', fontsize=12)
axes[1].set_title('CQL: Policy Quality vs Alpha', fontsize=12)
axes[1].set_ylim(0, 1.1)
axes[1].legend(); axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('/tmp/offline_rl_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## Key Takeaways

**Core idea:** Offline RL trains from a fixed dataset without environment interaction.
The core challenge is Q-value overestimation on OOD (unobserved) state-action pairs
caused by Bellman bootstrapping. CQL prevents this by actively pulling OOD Q-values
toward zero, making the policy conservative on unseen regions.

**Variants and when to use:**

| Method | OOD handling | Data efficiency | Online needed | Use when |
|--------|-------------|-----------------|---------------|----------|
| BC | No (ignores reward) | High | No | Behavior near-optimal |
| Standard QL offline | No (diverges) | Low | No | Avoid: overestimates Q |
| CQL | Yes (conservative) | Medium | No | Fixed dataset, no env access |
| CQL + online FT | Yes + adaptive | High | Yes | Best of both: offline+online |
| IQL | Yes (implicit) | Medium | No | Large datasets, stable training |

**Common failure modes:**
- Standard Q offline: overestimates Q on OOD actions -> poor policy
- CQL alpha too high: over-conservative, collapses toward BC performance
- Coverage gap: BC quality bounded by behavior policy's quality


## Exercises

1. **Coverage experiment**: Vary epsilon in `collect_offline_dataset` from 0 to 1.
   Plot coverage ratio vs CQL success rate. Where is the quality-coverage tradeoff?
2. **CQL alpha sensitivity**: Run the CQL alpha sweep with more values [0, 0.01, ..., 1.0].
   Find the optimal alpha. Does it change when dataset size doubles?
3. **Online budget**: Halve online steps to 250. How much does CQL pretraining
   help relative to from-scratch? When does pretraining stop helping?
4. **OOD quantification**: After training both Q_std and Q_cql, count how many
   (s,a) pairs are overestimated (Q > Q*). Which method has fewer? By how much?
